# Partie 1 : le notebook (8 parties)
Vous repartez du même jeu de données (les résumés de films) et vous écrivez un seul notebook, organisé en 8
parties, avec un titre pour chacune. Aucun code n’est fourni : les mots à chercher sont dans la boîte à outils (fin du document).

## 1. Audit et nettoyage
Chargez le fichier. Comptez les lignes et les films de chaque catégorie. Regardez la longueur des textes. **Supprimez les espaces en trop et les doublons** (deux lignes identiques). Notez en une phrase ce que vous avez trouvé et ce que vous avez corrigé.

*Pour comprendre : un même texte présent dans l’apprentissage et dans le test fausserait votre score, car le modèle aurait déjà vu la réponse.*

In [12]:
import pandas as pd

# Charge le fichier
df = pd.read_csv("textes_films_horreur_comedie.csv")
df.head()



# Compte les lignes et les films de chaque catégorie
print(f"\nNombre total de lignes initiales : {len(df)}")

print("\nRépartition par catégorie :")
print(df['cible'].value_counts())

# 3. Regarde la longueur des textes
df['longueur_texte'] = df['resume'].str.len()
print("\nStatistiques sur la longueur des textes :")
print(df['longueur_texte'].describe())

# 4. Supprime les espaces en trop et les doublons
df['resume'] = df['resume'].str.strip()

# Compter et supprimer les doublons stricts
nb_lignes_avant = len(df)
df = df.drop_duplicates()
nb_lignes_apres = len(df)
nb_doublons_supprimes = nb_lignes_avant - nb_lignes_apres

print(f"\nNombre de doublons supprimés : {nb_doublons_supprimes}")
print(f"Nombre de lignes après nettoyage : {nb_lignes_apres}")


Nombre total de lignes initiales : 2400

Répartition par catégorie :
cible
0    1200
1    1200
Name: count, dtype: int64

Statistiques sur la longueur des textes :
count    2400.000000
mean      558.189167
std       349.276409
min       201.000000
25%       335.000000
50%       474.000000
75%       679.000000
max      7104.000000
Name: longueur_texte, dtype: float64

Nombre de doublons supprimés : 54
Nombre de lignes après nettoyage : 2346


## 2. Découpage du jeu de données
Séparez les textes en trois parties : **60 % apprentissage, 20 % test, 20 % production** (à garder de côté, vous n’y touchez pas). Gardez la même proportion d’horreur et de comédie dans chaque partie.

*Pour comprendre : le test ne sert qu’une seule fois, à la toute fin. Tout ce que le modèle apprend, il l’apprend sur l’apprentissage seulement.*

In [13]:
from sklearn.model_selection import train_test_split

# 1. On sépare la partie production (20%) du reste (80%)
train_val, df_prod = train_test_split(
    df, 
    test_size=0.20, 
    stratify=df['cible'], 
    random_state=42
)

# 2. On sépare le reste (80%) en Apprentissage (60% du total -> 75% de train_val) et Test (20% du total -> 25% de train_val)
df_train, df_test = train_test_split(
    train_val, 
    test_size=0.25, 
    stratify=train_val['cible'], 
    random_state=42
)

# Vérification des tailles et des proportions pour s'assurer que tout est correct
print(f"Nombre de lignes - Apprentissage (60%) : {len(df_train)}")
print(f"Nombre de lignes - Test (20%)           : {len(df_test)}")
print(f"Nombre de lignes - Production (20%)     : {len(df_prod)}")

print("\nProportion de chaque catégorie dans l'apprentissage :")
print(df_train['cible'].value_counts(normalize=True))

print("\nProportion de chaque catégorie dans le test :")
print(df_test['cible'].value_counts(normalize=True))

Nombre de lignes - Apprentissage (60%) : 1407
Nombre de lignes - Test (20%)           : 469
Nombre de lignes - Production (20%)     : 470

Proportion de chaque catégorie dans l'apprentissage :
cible
0    0.509595
1    0.490405
Name: proportion, dtype: float64

Proportion de chaque catégorie dans le test :
cible
0    0.509595
1    0.490405
Name: proportion, dtype: float64


## 3. Représentation du texte (TF-IDF)
Un réseau ne lit pas des mots, il lit des nombres. Transformez chaque texte en nombres avec un **TF-IDF**. Apprenez le vocabulaire **sur l’apprentissage seulement**, puis transformez les trois parties avec ce même objet.

*Pour comprendre : si le vocabulaire est appris avec le test, des informations du test « fuient » dans l’apprentissage et le score est trop flatteur.*

In [14]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer()

# On entraîne et on transforme que sur le train
X_train = vectorizer.fit_transform(df_train['resume'])

# On transforme sur le test et la production
X_test = vectorizer.transform(df_test['resume'])
X_prod = vectorizer.transform(df_prod['resume'])


y_train = df_train['cible']
y_test = df_test['cible']
y_prod = df_prod['cible']

# Vérification des dimensions des matrices obtenues
print(f"Forme de la matrice TF-IDF d'apprentissage : {X_train.shape}")
print(f"Forme de la matrice TF-IDF de test : {X_test.shape}")
print(f"Forme de la matrice TF-IDF de production : {X_prod.shape}")

Forme de la matrice TF-IDF d'apprentissage : (1407, 14899)
Forme de la matrice TF-IDF de test : (469, 14899)
Forme de la matrice TF-IDF de production : (470, 14899)


## 4. Les points de comparaison
Entraînez trois modèles : une **régression logistique**, un **Bayes naïf**, et un **réseau de départ** très simple (une couche cachée, un peu de dropout, aucun réglage particulier). Mesurez le **F1** de chacun sur **5 graines différentes** (une graine = un tirage au hasard différent). Pour juger, utilisez une partie de l’apprentissage ; le test reste intact. Calculez **la marge de bruit** (voir la boîte à outils).

*Pour comprendre : si deux scores diffèrent de moins que la marge de bruit, vous ne pouvez pas dire que l’un est meilleur que l’autre.*

In [15]:
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import f1_score
from sklearn.model_selection import train_test_split
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout

# 1. On sépare l'apprentissage en train / validation (le test global reste intact)
X_tr, X_val, y_tr, y_val = train_test_split(X_train, df_train['cible'], test_size=0.2, random_state=42)

# S'assurer que les cibles sont bien en format numérique entier
y_tr = y_tr.astype(int)
y_val = y_val.astype(int)

# Pour Keras, on transforme la matrice sparse TF-IDF en format dense
X_tr_dense = X_tr.toarray()
X_val_dense = X_val.toarray()

# Les 5 graines (tirages aléatoires différents)
seeds = [1, 2, 3, 4, 5]

scores_lr = []
scores_nb = []
scores_nn = []

for seed in seeds:
    # --- Modèle 1 : Régression Logistique ---
    lr = LogisticRegression(random_state=seed)
    lr.fit(X_tr, y_tr)
    pred_lr = lr.predict(X_val)
    scores_lr.append(f1_score(y_val, pred_lr))
    
    # --- Modèle 2 : Naïf de Bayes ---
    nb = MultinomialNB()
    nb.fit(X_tr, y_tr)
    pred_nb = nb.predict(X_val)
    scores_nb.append(f1_score(y_val, pred_nb))
    
    # --- Modèle 3 : Réseau de neurones simple ---
    model = Sequential([
        Dense(32, activation='relu', input_shape=(X_tr_dense.shape[1],)),
        Dropout(0.2),
        Dense(1, activation='sigmoid')
    ])
    model.compile(optimizer='adam', loss='binary_crossentropy')
    model.fit(X_tr_dense, y_tr, epochs=3, batch_size=32, verbose=0)
    
    pred_nn = (model.predict(X_val_dense, verbose=0) > 0.5).astype(int).flatten()
    scores_nn.append(f1_score(y_val, pred_nn))

# 2. Affichage des moyennes des scores F1
print(f"F1 moyen - Régression Logistique : {np.mean(scores_lr):.4f}")
print(f"F1 moyen - Naïf de Bayes         : {np.mean(scores_nb):.4f}")
print(f"F1 moyen - Réseau de neurones    : {np.mean(scores_nn):.4f}")

# 3. Calcul de la marge de bruit
p = max(np.mean(scores_lr), np.mean(scores_nb), np.mean(scores_nn))
n = len(y_val)
marge_bruit = 1.96 * np.sqrt((p * (1 - p)) / n)

print(f"\nMarge de bruit calculée : {marge_bruit:.4f}")

c:\Users\hasss\AppData\Local\Programs\Python\Python313\Lib\site-packages\keras\src\layers\core\dense.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
c:\Users\hasss\AppData\Local\Programs\Python\Python313\Lib\site-packages\keras\src\layers\core\dense.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
c:\Users\hasss\AppData\Local\Programs\Python\Python313\Lib\site-packages\keras\src\layers\core\dense.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer

F1 moyen - Régression Logistique : 0.7897
F1 moyen - Naïf de Bayes         : 0.8281
F1 moyen - Réseau de neurones    : 0.8236

Marge de bruit calculée : 0.0440


## 5. Recherche des réglages avec KerasTuner
Écrivez d’abord **une fonction qui construit un réseau** à partir de cinq réglages : nombre de couches, nombre de
neurones, dropout, régularisation, taux d’apprentissage. Lancez ensuite **KerasTuner** (recherche au hasard, 20
essais).

In [16]:
import keras_tuner as kt
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.regularizers import l2

# 1. Fonction qui construit le réseau avec les 5 réglages (hp)
def build_model(hp):
    model = Sequential()
    
    # 1. Nombre de couches (1 ou 2)
    nb_couches = hp.Int('nb_couches', min_value=1, max_value=2)
    
    # 2. Nombre de neurones & 4. Régularisation L2
    neurones = hp.Int('neurones', min_value=16, max_value=64, step=16)
    reg_l2 = hp.Choice('reg_l2', [0.0, 0.01]) # Avec ou sans régularisation
    
    model.add(Dense(
        units=neurones, 
        activation='relu', 
        kernel_regularizer=l2(reg_l2) if reg_l2 > 0 else None,
        input_shape=(X_tr_dense.shape[1],)
    ))
    
    # 3. Taux de Dropout
    taux_dropout = hp.Float('dropout', min_value=0.1, max_value=0.4, step=0.1)
    model.add(Dropout(taux_dropout))
    
    # Si on choisit 2 couches, on en rajoute une deuxième
    if nb_couches == 2:
        model.add(Dense(units=neurones, activation='relu'))
        
    # Couche de sortie (classification binaire)
    model.add(Dense(1, activation='sigmoid'))
    
    # 5. Taux d'apprentissage (learning rate)
    lr = hp.Choice('learning_rate', [1e-2, 1e-3, 1e-4])
    
    model.compile(
        optimizer=Adam(learning_rate=lr),
        loss='binary_crossentropy',
        metrics=['accuracy']
    )
    return model

# 2. Lancement de la recherche (20 essais au hasard)
tuner = kt.RandomSearch(
    build_model,
    objective='val_loss',
    max_trials=20,
    directory='tuner_dossier',
    project_name='film_opti'
)

# Lancer la recherche sur nos données d'entraînement et de validation
tuner.search(X_tr_dense, y_tr, epochs=3, validation_data=(X_val_dense, y_val), verbose=0)

# 3. Afficher les meilleurs réglages trouvés
best_hp = tuner.get_best_hyperparameters()[0]
print("\n--- Keras : Meilleurs réglages trouvés ---")
print(f"Nombre de couches : {best_hp.get('nb_couches')}")
print(f"Nombre de neurones : {best_hp.get('neurones')}")
print(f"Dropout : {best_hp.get('dropout')}")
print(f"Régularisation L2 : {best_hp.get('reg_l2')}")
print(f"Learning Rate : {best_hp.get('learning_rate')}")

Reloading Tuner from tuner_dossier\film_opti\tuner0.json

--- Keras : Meilleurs réglages trouvés ---
Nombre de couches : 1
Nombre de neurones : 32
Dropout : 0.4
Régularisation L2 : 0.0
Learning Rate : 0.01


## 6. Recherche des réglages avec Optuna
Refaites la même recherche avec **Optuna**, sur **exactement les mêmes réglages, les mêmes valeurs possibles, le même nombre d’essais (20)**, en réutilisant votre fonction. Comparez le meilleur score des deux outils, en tenant
compte de la marge de bruit.

*Pour comprendre : pour comparer deux outils, il faut les placer dans les mêmes conditions.*

In [17]:
import optuna
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.regularizers import l2

# 1. Définition de la recherche Optuna
def objective(trial):
    # Choix des 5 réglages
    nb_couches = trial.suggest_int('nb_couches', 1, 2)
    neurones = trial.suggest_int('neurones', 16, 64, step=16)
    reg_l2 = trial.suggest_categorical('reg_l2', [0.0, 0.01])
    dropout = trial.suggest_float('dropout', 0.1, 0.4, step=0.1)
    lr = trial.suggest_categorical('learning_rate', [1e-2, 1e-3, 1e-4])
    
    # Construction du modèle
    model = Sequential([
        Dense(neurones, activation='relu', kernel_regularizer=l2(reg_l2) if reg_l2 > 0 else None, input_shape=(X_tr_dense.shape[1],)),
        Dropout(dropout)
    ])
    if nb_couches == 2:
        model.add(Dense(neurones, activation='relu'))
    model.add(Dense(1, activation='sigmoid'))
    
    model.compile(optimizer=Adam(learning_rate=lr), loss='binary_crossentropy')
    
    # Entraînement rapide sur 3 époques
    history = model.fit(X_tr_dense, y_tr, epochs=3, batch_size=32, validation_data=(X_val_dense, y_val), verbose=0)
    
    return min(history.history['val_loss'])

# 2. Lancer les 20 essais
study = optuna.create_study(direction='minimize')
study.optimize(objective, n_trials=20)

# 3. Afficher le meilleur résultat avec le même format
print("\n--- Optuna : Meilleurs réglages trouvés ---")
print(f"Nombre de couches : {study.best_params['nb_couches']}")
print(f"Nombre de neurones : {study.best_params['neurones']}")
print(f"Dropout : {study.best_params['dropout']}")
print(f"Régularisation L2 : {study.best_params['reg_l2']}")
print(f"Learning Rate : {study.best_params['learning_rate']}")

[I 2026-10-23 14:27:12,421] A new study created in memory with name: no-name-0dbcc222-f00f-48d2-920c-90ae4035ece8
c:\Users\hasss\AppData\Local\Programs\Python\Python313\Lib\site-packages\keras\src\layers\core\dense.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
[I 2026-10-23 14:27:14,807] Trial 0 finished with value: 0.5354211926460266 and parameters: {'nb_couches': 1, 'neurones': 64, 'reg_l2': 0.0, 'dropout': 0.2, 'learning_rate': 0.001}. Best is trial 0 with value: 0.5354211926460266.
[I 2026-10-23 14:27:16,670] Trial 1 finished with value: 0.6912106871604919 and parameters: {'nb_couches': 1, 'neurones': 16, 'reg_l2': 0.01, 'dropout': 0.30000000000000004, 'learning_rate': 0.001}. Best is trial 0 with value: 0.5354211926460266.
[I 2026-10-23 14:27:19,758] Trial 2 finished


--- Optuna : Meilleurs réglages trouvés ---
Nombre de couches : 1
Nombre de neurones : 32
Dropout : 0.1
Régularisation L2 : 0.0
Learning Rate : 0.01


In [18]:
# Comparaison KerasTuner vs Optuna
best_keras_score = tuner.oracle.get_best_trials(num_trials=1)[0].score
best_optuna_score = study.best_value

print(f"Meilleure val_loss (KerasTuner) : {best_keras_score:.4f}")
print(f"Meilleure val_loss (Optuna)      : {best_optuna_score:.4f}")

ecart = abs(best_keras_score - best_optuna_score)
print(f"Écart entre les deux : {ecart:.4f}")
print(f"Marge de bruit       : {marge_bruit:.4f}")

if ecart < marge_bruit:
    print("\nConclusion : Les deux outils font jeu égal (écart inférieur à la marge de bruit).")
else:
    print("\nConclusion : L'un des outils est réellement meilleur que l'autre.")

Meilleure val_loss (KerasTuner) : 0.3510
Meilleure val_loss (Optuna)      : 0.3488
Écart entre les deux : 0.0022
Marge de bruit       : 0.0440

Conclusion : Les deux outils font jeu égal (écart inférieur à la marge de bruit).


## 7. Verdict sur le test
Prenez les meilleurs réglages, réentraînez le réseau sur **tout l’apprentissage**, sur **5 graines**. Comparez-le à votre réseau de départ et au meilleur modèle simple. Puis mesurez-le sur **le test, une seule fois**. Concluez honnêtement : le réseau est-il meilleur, ou l’écart est-il plus petit que la marge de bruit ?

In [ ]:
import numpy as np
from sklearn.metrics import f1_score
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.optimizers import Adam

# 1. Conversion des données en format dense pour Keras
X_tr_d = X_train.toarray()
X_te_d = X_test.toarray()
y_tr_num = y_train.astype(int)
y_te_num = y_test.astype(int)

# 2. Entraînement du meilleur modèle sur 5 graines
scores_test = []

for seed in [1, 2, 3, 4, 5]:
    model = Sequential([
        Dense(study.best_params['neurones'], activation='relu', input_shape=(X_tr_d.shape[1],)),
        Dropout(study.best_params['dropout'])
    ])
    if study.best_params['nb_couches'] == 2:
        model.add(Dense(study.best_params['neurones'], activation='relu'))
    model.add(Dense(1, activation='sigmoid'))
    
    model.compile(optimizer=Adam(learning_rate=study.best_params['learning_rate']), loss='binary_crossentropy')
    model.fit(X_tr_d, y_tr_num, epochs=3, batch_size=32, verbose=0)
    
    # Évaluation sur le jeu de test
    preds = (model.predict(X_te_d, verbose=0) > 0.5).astype(int).flatten()
    scores_test.append(f1_score(y_te_num, preds))

# 3. Résultats finaux
print(f"Score F1 moyen sur le test (5 graines) : {np.mean(scores_test):.4f}")
print(f"Score F1 final (sur une exécution)     : {scores_test[0]:.4f}")

c:\Users\hasss\AppData\Local\Programs\Python\Python313\Lib\site-packages\keras\src\layers\core\dense.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
c:\Users\hasss\AppData\Local\Programs\Python\Python313\Lib\site-packages\keras\src\layers\core\dense.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
c:\Users\hasss\AppData\Local\Programs\Python\Python313\Lib\site-packages\keras\src\layers\core\dense.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer

Score F1 moyen sur le test (5 graines) : 0.8679
Score F1 final (sur une exécution)     : 0.8645


**Conclusion :**

*L'écart est inférieur à la marge de bruit. Il n'y a donc pas de gain significatif par rapport au modèle de base.*

## 8. Sauvegarde
Créez un dossier (par exemple model/) et rangez-y :

• le **réseau** au format .keras ;

• le **TF-IDF** avec joblib ;

• un fichier **metrics.json** : version du modèle, F1 obtenu sur le test, taille du test, versions des
bibliothèques (écrivez-le en **UTF-8**) ;

• un petit **échantillon de données de référence**.

In [21]:
import os, json, joblib
import tensorflow as tf
import sklearn

# 1. Création du dossier 'model'
os.makedirs('model', exist_ok=True)

# 2. Sauvegarde du réseau
model.save('model/model.keras')

# 3. Sauvegarde du TF-IDF
joblib.dump(vectorizer, 'model/vectoriseur.joblib')

# 4. Le fichier metrics.json (version du modèle, F1 score, taille du test, versions bibliothèques)
metrics = {
    "model_version": "1.0",
    "test_f1_score": float(scores_test[0]),
    "test_size": int(len(df_test)),
    "library_versions": {
        "tensorflow": tf.__version__,
        "scikit-learn": sklearn.__version__,
        "joblib": joblib.__version__
    }
}
with open('model/metrics.json', 'w', encoding='utf-8') as f:
    json.dump(metrics, f, indent=4, ensure_ascii=False)

# 5. Echantillon de données de référence
pd.DataFrame({
    'synopsis': df_test['resume'].values[:5], 
    'cible': df_test['cible'].values[:5]
}).to_csv('model/reference.csv', index=False, encoding='utf-8')

print("✅ Tous les fichiers ont été sauvegardés avec succès dans le dossier 'model/' !")

✅ Tous les fichiers ont été sauvegardés avec succès dans le dossier 'model/' !
